In [1]:
# Cell 1: Install dependencies
!pip install -q pymupdf opencv-python-headless pillow pandas tqdm transformers torch torchvision open_clip_torch
!pip install -q paddlepaddle paddleocr "paddlex[ocr]==3.7.2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 66.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 56.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 26.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 8.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 8.7 MB/s eta 0:00:00
   ━━━

In [2]:
# Cell 2: Imports
import os
import re
import json
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pymupdf
import torch
import open_clip
from PIL import Image
from tqdm import tqdm
from IPython.display import display


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [3]:
# Cell 3: Configuration for Kaggle
PDF_PATH = "/kaggle/input/datasets/boopaland3v/sample-pdf-data/The_European_Medical_Device_Regulation-SECURED.pdf"

WORK_DIR = Path("/kaggle/working/flowchart_detection_v6")
EMBEDDED_DIR = WORK_DIR / "01_embedded_images"
RENDERED_DIR = WORK_DIR / "02_rendered_vector_pages"
TABLE_DEBUG_DIR = WORK_DIR / "03_table_debug"

# Rendering / filtering
RENDER_SCALE = 2
MIN_DRAWINGS_TO_RENDER = 20
MIN_WIDTH = 250
MIN_HEIGHT = 180
MIN_AREA = 60_000

# Performance controls
MAX_LAYOUT_INPUTS = 160        # Paddle layout is cheaper than OpenCLIP. Keep this reasonably high.
MAX_OPENCLIP_INPUTS = 100      # OpenCLIP should see only the stronger survivors.
MAX_FLORENCE_ITEMS = 8         # Final fallback should stay tiny for speed.
USE_PADDLE_LAYOUT = True
USE_FLORENCE_FALLBACK = True
FLORENCE_MODEL_ID = "microsoft/Florence-2-base-ft"

OUTPUT_FOLDERS = [EMBEDDED_DIR, RENDERED_DIR, TABLE_DEBUG_DIR]
for folder in OUTPUT_FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

print("PDF path:", PDF_PATH)
print("Work dir:", WORK_DIR)
print("Florence fallback:", USE_FLORENCE_FALLBACK, "| model:", FLORENCE_MODEL_ID)


PDF path: /kaggle/input/datasets/boopaland3v/sample-pdf-data/The_European_Medical_Device_Regulation-SECURED.pdf
Work dir: /kaggle/working/flowchart_detection_v6
Florence fallback: True | model: microsoft/Florence-2-base-ft


In [4]:
# Cell 4: Clean workspace for every full run
for folder in OUTPUT_FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)
    for item in folder.iterdir():
        if item.is_file():
            item.unlink()
        elif item.is_dir():
            shutil.rmtree(item)

print("Workspace cleaned.")
for folder in OUTPUT_FOLDERS:
    print(f"{folder.name:30s}: {len(list(folder.iterdir()))} files")


Workspace cleaned.
01_embedded_images            : 0 files
02_rendered_vector_pages      : 0 files
03_table_debug                : 0 files


In [5]:
# Cell 5: Utility helpers

FLOW_WORDS = [
    "flowchart", "flow chart", "workflow", "process", "procedure", "decision",
    "start", "end", "yes", "no", "define", "manufacturer flow", "device flow",
    "submit", "approval", "classify", "covered", "excluded", "continue"
]

TABLE_WORDS = [
    "table", "reference map", "article numbers", "annex numbers", "key:",
    "row", "column", "contents", "index", "list of tables", "chapter no",
    "this part is referred to by", "this part also refers to"
]

NEGATIVE_VISUAL_WORDS = [
    "logo", "cover", "guidebook", "series", "regulatory affairs professionals society",
    "meddev solutions", "reference map"
]


def safe_image_size(path):
    try:
        img = Image.open(path)
        w, h = img.size
        return int(w), int(h), int(w * h)
    except Exception:
        return 0, 0, 0


def count_keywords(text, words):
    text = str(text).lower()
    return sum(1 for w in words if w in text)


def page_text_features(page):
    text = page.get_text("text") or ""
    text_l = text.lower()
    return {
        "page_text_preview": text[:500],
        "flow_word_count": count_keywords(text_l, FLOW_WORDS),
        "table_word_count": count_keywords(text_l, TABLE_WORDS),
        "negative_word_count": count_keywords(text_l, NEGATIVE_VISUAL_WORDS),
        "has_flow_words": count_keywords(text_l, FLOW_WORDS) > 0,
        "has_table_words": count_keywords(text_l, TABLE_WORDS) > 0,
    }


def analyze_pdf_vectors(page):
    drawings = page.get_drawings()
    horizontal_lines = 0
    vertical_lines = 0
    rectangles = 0
    thin_lines = 0

    for d in drawings:
        rect = d.get("rect")
        if rect is None:
            continue
        w, h = float(rect.width), float(rect.height)
        if w > 40 and h < 3:
            horizontal_lines += 1
            thin_lines += 1
        if h > 40 and w < 3:
            vertical_lines += 1
            thin_lines += 1
        if w > 20 and h > 10:
            rectangles += 1

    table_grid_score = 0.0
    if horizontal_lines >= 15:
        table_grid_score += 0.25
    if vertical_lines >= 15:
        table_grid_score += 0.25
    if rectangles >= 35:
        table_grid_score += 0.30
    if thin_lines >= 30:
        table_grid_score += 0.20
    table_grid_score = min(table_grid_score, 1.0)

    return {
        "drawing_count": len(drawings),
        "pdf_horizontal_lines": horizontal_lines,
        "pdf_vertical_lines": vertical_lines,
        "pdf_rectangles": rectangles,
        "pdf_thin_lines": thin_lines,
        "table_grid_score": table_grid_score,
        "is_pdf_table_like": table_grid_score >= 0.70,
    }


In [6]:
# Cell 6: PyMuPDF extraction + candidate normalization
# Every source becomes a candidate image:
# - embedded_image: extracted from PDF as bitmap
# - rendered_vector_page: rendered from PDF vector drawings
# Both types will pass through the same rule + geometry + OpenCLIP pipeline.

doc = pymupdf.open(PDF_PATH)
records = []

for page_index in tqdm(range(len(doc)), desc="Scanning PDF"):
    page = doc[page_index]
    page_number = page_index + 1
    images = page.get_images(full=True)
    vector_metrics = analyze_pdf_vectors(page)
    text_metrics = page_text_features(page)

    # 1) Extract embedded raster images
    for image_index, img in enumerate(images):
        xref = img[0]
        try:
            extracted = doc.extract_image(xref)
            image_ext = extracted.get("ext", "png")
            image_path = EMBEDDED_DIR / f"page_{page_number}_img_{image_index + 1}.{image_ext}"
            with open(image_path, "wb") as f:
                f.write(extracted["image"])
            w, h, area = safe_image_size(image_path)
            records.append({
                "page_number": page_number,
                "source_type": "embedded_image",
                "image_index": image_index + 1,
                "image_path": str(image_path),
                "width": w,
                "height": h,
                "area": area,
                "image_count": len(images),
                **vector_metrics,
                **text_metrics,
            })
        except Exception as e:
            print(f"Embedded image extraction failed on page {page_number}: {e}")

    # 2) Render vector-heavy pages
    if vector_metrics["drawing_count"] >= MIN_DRAWINGS_TO_RENDER:
        # Render to debug-table dir if the page is strongly table-like, otherwise rendered dir.
        output_dir = TABLE_DEBUG_DIR if vector_metrics["is_pdf_table_like"] else RENDERED_DIR
        output_dir.mkdir(parents=True, exist_ok=True)
        suffix = "table_like" if vector_metrics["is_pdf_table_like"] else "rendered"
        render_path = output_dir / f"page_{page_number}_{suffix}.png"
        try:
            pix = page.get_pixmap(matrix=pymupdf.Matrix(RENDER_SCALE, RENDER_SCALE), alpha=False)
            pix.save(str(render_path))
            w, h, area = safe_image_size(render_path)
            records.append({
                "page_number": page_number,
                "source_type": "rendered_vector_page",
                "image_index": 0,
                "image_path": str(render_path),
                "width": w,
                "height": h,
                "area": area,
                "image_count": len(images),
                **vector_metrics,
                **text_metrics,
            })
        except Exception as e:
            print(f"Rendering failed on page {page_number}: {e}")

df_sources = pd.DataFrame(records)
print("Total normalized candidate images:", len(df_sources))
print(df_sources["source_type"].value_counts(dropna=False))
df_sources[["page_number", "source_type", "width", "height", "drawing_count", "table_grid_score", "flow_word_count", "table_word_count"]].head(20)


Scanning PDF: 100%|██████████| 384/384 [00:28<00:00, 13.60it/s]

Total normalized candidate images: 296
source_type
rendered_vector_page    241
embedded_image           55
Name: count, dtype: int64


,page_number,source_type,width,height,drawing_count,table_grid_score,flow_word_count,table_word_count
0,1,embedded_image,2858,4038,90,0.00,0,0
1,1,rendered_vector_page,1191,1684,90,0.00,0,0
2,3,rendered_vector_page,1191,1684,87,0.00,0,0
3,14,rendered_vector_page,1191,1684,721,0.00,2,1
4,15,embedded_image,1646,3263,1,0.00,0,0
5,16,embedded_image,1650,3280,1,0.00,0,0
6,17,embedded_image,1575,2858,1,0.00,0,0
7,18,rendered_vector_page,1191,1684,44,0.70,2,2
8,19,rendered_vector_page,1191,1684,42,0.70,2,0
9,20,rendered_vector_page,1191,1684,37,0.25,1,0


In [7]:
# Cell 7: Unified pre-OpenCLIP rule funnel (robust barcode + generalized cover rejection)
# Runs on BOTH embedded images and rendered vector pages.
# Goal:
# 1) remove obvious junk
# 2) keep borderline flowchart-like pages alive
# 3) reduce OpenCLIP load without silently killing true flowcharts

def image_geometry_fast(image_path):
    img = cv2.imread(str(image_path))
    if img is None:
        return {
            "img_line_score": 0.0,
            "img_shape_score": 0.0,
            "img_table_score": 0.0,
            "img_horizontal_lines": 0,
            "img_vertical_lines": 0,
            "img_contours": 0,
            "img_rect_like": 0,
            "img_diamond_like": 0,
            "img_ellipse_like": 0,
            "img_arrow_like": 0,
            "img_dark_ratio": 0.0,
            "img_barcode_like_score": 0.0,
            "img_edge_density": 0.0,
        }

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)

    binary = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 15
    )

    h_img, w_img = gray.shape
    img_area = h_img * w_img

    # Dark pixel ratio
    img_dark_ratio = float((gray < 80).sum()) / float(img_area)

    # Edge density: covers/photos often have smoother large regions,
    # while diagrams/tables/barcodes have stronger edge structure
    edges = cv2.Canny(gray, 80, 180)
    img_edge_density = float((edges > 0).sum()) / float(img_area)

    # Horizontal / vertical line signals
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (45, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 45))

    horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel)
    vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vertical_kernel)

    h_contours, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    v_contours, _ = cv2.findContours(vertical, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    horizontal_lines = len(h_contours)
    vertical_lines = len(v_contours)

    # Shape-like contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    rect_like = 0
    diamond_like = 0
    ellipse_like = 0
    arrow_like = 0

    for c in contours:
        area = cv2.contourArea(c)
        if area < img_area * 0.0003 or area > img_area * 0.35:
            continue

        peri = cv2.arcLength(c, True)
        if peri <= 0:
            continue

        approx = cv2.approxPolyDP(c, 0.04 * peri, True)
        x, y, w, h = cv2.boundingRect(c)
        aspect = w / max(h, 1)

        if len(approx) == 4:
            if 0.75 <= aspect <= 1.35:
                diamond_like += 1
            else:
                rect_like += 1
        elif len(approx) >= 6 and 0.5 <= aspect <= 2.2:
            ellipse_like += 1

        # connector / arrow proxy
        if (w > 25 or h > 25) and area < img_area * 0.02:
            arrow_like += 1

    img_line_score = min((horizontal_lines + vertical_lines) / 40.0, 1.0)
    img_shape_score = min(
        (rect_like * 0.08) +
        (diamond_like * 0.20) +
        (ellipse_like * 0.12) +
        (arrow_like * 0.03),
        1.0
    )

    # Table-like proxy
    img_table_score = 0.0
    if horizontal_lines >= 12:
        img_table_score += 0.30
    if vertical_lines >= 12:
        img_table_score += 0.30
    if rect_like >= 30:
        img_table_score += 0.25
    if diamond_like == 0 and ellipse_like == 0:
        img_table_score += 0.15
    img_table_score = min(img_table_score, 1.0)

    # Barcode-like proxy
    img_barcode_like_score = 0.0
    if vertical_lines >= 25:
        img_barcode_like_score += 0.45
    if horizontal_lines <= 3:
        img_barcode_like_score += 0.15
    if diamond_like == 0 and ellipse_like == 0:
        img_barcode_like_score += 0.15
    if arrow_like < 2:
        img_barcode_like_score += 0.10
    if img_dark_ratio >= 0.08:
        img_barcode_like_score += 0.15
    img_barcode_like_score = min(img_barcode_like_score, 1.0)

    return {
        "img_line_score": img_line_score,
        "img_shape_score": img_shape_score,
        "img_table_score": img_table_score,
        "img_horizontal_lines": horizontal_lines,
        "img_vertical_lines": vertical_lines,
        "img_contours": len(contours),
        "img_rect_like": rect_like,
        "img_diamond_like": diamond_like,
        "img_ellipse_like": ellipse_like,
        "img_arrow_like": arrow_like,
        "img_dark_ratio": img_dark_ratio,
        "img_barcode_like_score": img_barcode_like_score,
        "img_edge_density": img_edge_density,
    }


# ---- Run fast geometry on all candidates ----
fast_records = []
for _, row in tqdm(df_sources.iterrows(), total=len(df_sources), desc="Fast image geometry"):
    fast_records.append({
        **row.to_dict(),
        **image_geometry_fast(row["image_path"])
    })

df_fast = pd.DataFrame(fast_records)


def pre_rule_decision(row):
    """
    High-recall pre-OpenCLIP funnel.
    Returns one of:
      - reject_small
      - reject_barcode_like
      - reject_cover_or_logo_words
      - reject_cover_like
      - reject_strong_table_text
      - reject_strong_table_grid
      - send_to_openclip
      - needs_light_review
      - reject_low_signal
    """

    page_number = row.get("page_number", 9999)
    is_early_page = page_number <= 3

    # -------------------------
    # Hard reject: tiny items
    # -------------------------
    if row["width"] < MIN_WIDTH or row["height"] < MIN_HEIGHT or row["area"] < MIN_AREA:
        return "reject_small"

    # -------------------------
    # Barcode-like reject
    # -------------------------
    barcode_like = (
        row.get("img_barcode_like_score", 0) >= 0.70 and
        row.get("flow_word_count", 0) == 0 and
        row.get("img_shape_score", 0) < 0.20
    )
    if barcode_like:
        return "reject_barcode_like"

    # -------------------------
    # Obvious cover/logo words
    # -------------------------
    if row.get("negative_word_count", 0) >= 2 and row.get("flow_word_count", 0) == 0:
        if row.get("img_shape_score", 0) < 0.15 and row.get("img_arrow_like", 0) < 2:
            return "reject_cover_or_logo_words"

    # -------------------------
    # Generalized cover / branding image reject
    # Early-page prior + almost zero flow evidence
    # -------------------------
    cover_like = (
        is_early_page and
        row.get("img_shape_score", 0) < 0.12 and
        row.get("img_arrow_like", 0) < 2 and
        row.get("img_diamond_like", 0) == 0 and
        row.get("img_ellipse_like", 0) == 0 and
        row.get("flow_word_count", 0) == 0 and
        row.get("img_edge_density", 0) < 0.08
    )

    if cover_like:
        return "reject_cover_like"

    # -------------------------
    # Strong table checks
    # -------------------------
    strong_table = max(
        row.get("table_grid_score", 0),
        row.get("img_table_score", 0)
    ) >= 0.85

    table_text = row.get("table_word_count", 0) >= 2
    flow_text = row.get("flow_word_count", 0) >= 1
    has_flow_shapes = (
        row.get("img_diamond_like", 0) >= 1 or
        row.get("img_ellipse_like", 0) >= 1
    )
    has_flow_connectors = row.get("img_arrow_like", 0) >= 3

    if strong_table and table_text and not flow_text and not has_flow_shapes and not has_flow_connectors:
        return "reject_strong_table_text"

    if strong_table and not has_flow_shapes and row.get("img_arrow_like", 0) < 3 and row.get("flow_word_count", 0) == 0:
        return "reject_strong_table_grid"

    # -------------------------
    # Pre-OpenCLIP ranking score
    # -------------------------
    score = 0.0
    score += 0.35 * row.get("img_shape_score", 0)
    score += 0.20 * row.get("img_line_score", 0)
    score += 0.20 if row.get("flow_word_count", 0) > 0 else 0.0
    score += 0.15 if row.get("img_diamond_like", 0) > 0 else 0.0
    score += 0.10 if row.get("img_arrow_like", 0) >= 3 else 0.0

    # penalties
    score -= 0.25 * max(row.get("table_grid_score", 0), row.get("img_table_score", 0))

    if row.get("table_word_count", 0) > row.get("flow_word_count", 0):
        score -= 0.10

    if row.get("negative_word_count", 0) >= 2 and row.get("flow_word_count", 0) == 0:
        score -= 0.08

    if row.get("img_barcode_like_score", 0) >= 0.50:
        score -= 0.20

    # early-page non-diagram penalty (soft, not hard)
    if is_early_page and row.get("flow_word_count", 0) == 0 and row.get("img_shape_score", 0) < 0.15:
        score -= 0.08

    # -------------------------
    # Safer routing
    # -------------------------
    if score >= 0.10:
        return "send_to_openclip"
    elif score >= -0.05:
        return "needs_light_review"
    else:
        return "reject_low_signal"


df_fast["pre_rule_decision"] = df_fast.apply(pre_rule_decision, axis=1)

# Split buckets
df_openclip_pool = df_fast[df_fast["pre_rule_decision"] == "send_to_openclip"].copy()
df_light_review = df_fast[df_fast["pre_rule_decision"] == "needs_light_review"].copy()
df_pre_rejected = df_fast[
    ~df_fast["pre_rule_decision"].isin(["send_to_openclip", "needs_light_review"])
].copy()

# Priority score for OpenCLIP ranking
df_openclip_pool["preclip_candidate_score"] = (
    0.35 * df_openclip_pool["img_shape_score"] +
    0.20 * df_openclip_pool["img_line_score"] +
    0.20 * (df_openclip_pool["flow_word_count"] > 0).astype(float) +
    0.15 * (df_openclip_pool["img_diamond_like"] > 0).astype(float) +
    0.10 * (df_openclip_pool["img_arrow_like"] >= 3).astype(float) -
    0.25 * df_openclip_pool[["table_grid_score", "img_table_score"]].max(axis=1) -
    0.20 * df_openclip_pool["img_barcode_like_score"]
)

# Send only top-N to OpenCLIP immediately
df_clip_input = (
    df_openclip_pool
    .sort_values("preclip_candidate_score", ascending=False)
    .head(MAX_OPENCLIP_INPUTS)
    .copy()
)

# Anything left from send_to_openclip is deferred, NOT rejected
df_deferred_before_openclip = df_openclip_pool.drop(df_clip_input.index).copy()
if len(df_deferred_before_openclip) > 0:
    df_deferred_before_openclip["pre_rule_decision"] = "deferred_before_openclip"

# Final "not going to current OpenCLIP batch" bucket
df_pre_rejected = pd.concat(
    [df_pre_rejected, df_light_review, df_deferred_before_openclip],
    ignore_index=True
)

print("Total normalized candidates:", len(df_fast))
print("Hard pre-rejected / deferred / light-review:", len(df_pre_rejected))
print("Sent to OpenCLIP:", len(df_clip_input))
print(df_fast["pre_rule_decision"].value_counts())

display(
    df_clip_input[[
        "page_number",
        "source_type",
        "preclip_candidate_score",
        "pre_rule_decision",
        "img_shape_score",
        "img_table_score",
        "table_grid_score",
        "flow_word_count",
        "table_word_count",
        "negative_word_count",
        "img_diamond_like",
        "img_arrow_like",
        "img_vertical_lines",
        "img_horizontal_lines",
        "img_dark_ratio",
        "img_edge_density",
        "img_barcode_like_score",
    ]].head(20)
)

print("\nLight review bucket:", len(df_light_review))
print("Deferred before OpenCLIP:", len(df_deferred_before_openclip))

Fast image geometry: 100%|██████████| 296/296 [00:19<00:00, 14.96it/s]

Total normalized candidates: 296
Hard pre-rejected / deferred / light-review: 196
Sent to OpenCLIP: 100
pre_rule_decision
send_to_openclip      223
reject_small           35
needs_light_review     22
reject_low_signal      16
Name: count, dtype: int64


,page_number,source_type,preclip_candidate_score,pre_rule_decision,img_shape_score,img_table_score,table_grid_score,flow_word_count,table_word_count,negative_word_count,img_diamond_like,img_arrow_like,img_vertical_lines,img_horizontal_lines,img_dark_ratio,img_edge_density,img_barcode_like_score
279,357,rendered_vector_page,0.835,send_to_openclip,1.0,0.3,0.25,1,3,1,2,35,37,6,0.024276,0.044949,0.45
269,339,embedded_image,0.805,send_to_openclip,1.0,0.3,0.00,2,1,1,2,54,50,4,0.152397,0.073026,0.60
3,14,rendered_vector_page,0.760,send_to_openclip,1.0,0.6,0.00,2,1,1,4,25,44,29,0.019409,0.032866,0.45
134,168,rendered_vector_page,0.760,send_to_openclip,1.0,0.6,0.45,1,6,2,2,33,51,18,0.073240,0.054017,0.45
135,169,rendered_vector_page,0.760,send_to_openclip,1.0,0.6,0.45,1,6,2,2,36,51,18,0.070968,0.051736,0.45
113,132,rendered_vector_page,0.760,send_to_openclip,1.0,0.6,0.45,2,6,2,2,35,54,18,0.074370,0.056356,0.45
133,167,rendered_vector_page,0.760,send_to_openclip,1.0,0.6,0.45,2,6,2,1,40,63,18,0.074168,0.057408,0.45
150,189,rendered_vector_page,0.760,send_to_openclip,1.0,0.6,0.45,2,6,3,1,41,58,18,0.073930,0.056888,0.45
154,194,rendered_vector_page,0.760,send_to_openclip,1.0,0.6,0.45,1,6,2,1,43,58,18,0.071693,0.051419,0.45
145,182,rendered_vector_page,0.760,send_to_openclip,1.0,0.6,0.45,1,6,2,1,44,65,18,0.072171,0.052776,0.45



Light review bucket: 22
Deferred before OpenCLIP: 123


In [8]:
# Cell 8: Load Paddle layout detector
# This stage runs BEFORE OpenCLIP and removes obvious document-layout cases.

layout_pipeline = None

if USE_PADDLE_LAYOUT:
    try:
        from paddlex import create_pipeline
        layout_pipeline = create_pipeline(pipeline="layout_parsing")
        print("Paddle layout detector loaded successfully.")
    except Exception as e:
        print("Paddle layout detector unavailable. Continuing without it.")
        print("Reason:", e)
else:
    print("Skipping Paddle layout detector.")


Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

inference.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

inference.json: 0.00B [00:00, ?B/s]

inference.yml: 0.00B [00:00, ?B/s]

inference.pdiparams:   0%|          | 0.00/7.67M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

Paddle layout detector loaded successfully.


In [9]:
# Paddle Layout helper

def normalize_paddle_layout_output(result):
    """
    Normalize Paddle layout output into a list of dicts with at least:
    { "label": ..., "bbox": ... }
    """
    items = []

    if result is None:
        return items

    # case 1: already a list
    if isinstance(result, list):
        for obj in result:
            if isinstance(obj, dict):
                label = obj.get("label") or obj.get("type") or obj.get("category") or ""
                bbox = obj.get("bbox") or obj.get("box") or obj.get("points") or None
                items.append({"label": label, "bbox": bbox, "raw": obj})
        return items

    # case 2: dict-like
    if isinstance(result, dict):
        possible_keys = ["boxes", "layout", "results", "regions"]
        for key in possible_keys:
            if key in result and isinstance(result[key], list):
                for obj in result[key]:
                    if isinstance(obj, dict):
                        label = obj.get("label") or obj.get("type") or obj.get("category") or ""
                        bbox = obj.get("bbox") or obj.get("box") or obj.get("points") or None
                        items.append({"label": label, "bbox": bbox, "raw": obj})
                return items

    return items


def run_paddle_layout(image_path):
    """
    Runs Paddle layout detection on a single image and returns
    normalized layout items.
    Assumes a global object called `layout_pipeline` or `layout_model`
    exists from an earlier cell.
    """

    # use whichever object exists in your notebook
    model = None
    if "layout_pipeline" in globals():
        model = layout_pipeline
    elif "layout_model" in globals():
        model = layout_model
    else:
        raise NameError(
            "No Paddle layout model found. Expected `layout_pipeline` or `layout_model` "
            "to be created in a previous cell."
        )

    result = model.predict(str(image_path))
    return normalize_paddle_layout_output(result)

In [10]:
# Cell 10: Load OpenCLIP

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

OPENCLIP_MODEL = "ViT-B-32"
OPENCLIP_PRETRAINED = "laion2b_s34b_b79k"

model, _, preprocess = open_clip.create_model_and_transforms(
    OPENCLIP_MODEL,
    pretrained=OPENCLIP_PRETRAINED,
    device=DEVICE
)

tokenizer = open_clip.get_tokenizer(OPENCLIP_MODEL)
model.eval()

# Use one single label list variable consistently
OPENCLIP_LABELS = [
    "a classic flowchart diagram with boxes and arrows",
    "a workflow diagram showing process steps",
    "a decision tree diagram with branches",
    "a regulatory process flow diagram",
    "a process map with connected boxes and arrows",
    "a reference map or cross reference table",
    "a tabular document with rows and columns",
    "a document page with structured text",
    "a decorative logo or icon"
]

# Named label aliases
LABEL_FLOWCHART = OPENCLIP_LABELS[0]
LABEL_WORKFLOW = OPENCLIP_LABELS[1]
LABEL_DECISION_TREE = OPENCLIP_LABELS[2]
LABEL_REGULATORY_FLOW = OPENCLIP_LABELS[3]
LABEL_PROCESS_MAP = OPENCLIP_LABELS[4]
LABEL_REFERENCE_MAP = OPENCLIP_LABELS[5]
LABEL_TABLE = OPENCLIP_LABELS[6]
LABEL_TEXT_PAGE = OPENCLIP_LABELS[7]
LABEL_LOGO = OPENCLIP_LABELS[8]

# Encode label texts once
text_tokens = tokenizer(OPENCLIP_LABELS).to(DEVICE)

with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("OpenCLIP loaded with", len(OPENCLIP_LABELS), "labels")
print("Labels:")
for i, label in enumerate(OPENCLIP_LABELS):
    print(f"{i}: {label}")

Device: cpu


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

OpenCLIP loaded with 9 labels
Labels:
0: a classic flowchart diagram with boxes and arrows
1: a workflow diagram showing process steps
2: a decision tree diagram with branches
3: a regulatory process flow diagram
4: a process map with connected boxes and arrows
5: a reference map or cross reference table
6: a tabular document with rows and columns
7: a document page with structured text
8: a decorative logo or icon


In [11]:
# Cell 11: Paddle Layout interpretation + Run OpenCLIP

def interpret_layout_signals(layout_items):
    table_count = 0
    figure_count = 0
    chart_count = 0
    image_count = 0
    text_like_count = 0
    title_like_count = 0

    for item in layout_items:
        label = str(item.get("label", "")).lower()

        if "table" in label:
            table_count += 1
        elif "figure" in label:
            figure_count += 1
        elif "chart" in label:
            chart_count += 1
        elif "image" in label or "img" in label:
            image_count += 1
        elif label in ["text", "paragraph", "list", "caption", "reference", "footer", "header"]:
            text_like_count += 1
        elif label in ["title", "section", "heading"]:
            title_like_count += 1

    layout_table_signal = table_count
    layout_figure_signal = figure_count + chart_count + image_count
    layout_text_signal = text_like_count + title_like_count

    if layout_table_signal >= 2 and layout_figure_signal == 0:
        layout_decision = "table_like_penalty"
    elif layout_figure_signal >= 1:
        layout_decision = "diagram_like_boost"
    else:
        layout_decision = "neutral"

    return {
        "layout_table_signal": layout_table_signal,
        "layout_figure_signal": layout_figure_signal,
        "layout_text_signal": layout_text_signal,
        "layout_decision": layout_decision
    }


# ---------- run layout stage ----------
layout_records = []
for _, row in tqdm(df_clip_input.iterrows(), total=len(df_clip_input), desc="Running Paddle Layout"):
    try:
        layout_items = run_paddle_layout(row["image_path"]) if layout_pipeline is not None else []
    except Exception:
        layout_items = []

    signals = interpret_layout_signals(layout_items)

    layout_records.append({
        **row.to_dict(),
        **signals,
        "layout_raw": layout_items
    })

df_layout = pd.DataFrame(layout_records)
print("Layout stage completed:", len(df_layout))

# ---------- OpenCLIP ----------
def classify_openclip(image_path):
    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        image_features = model.encode_image(image)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)[0]

    scores = {
        OPENCLIP_LABELS[i]: float(probs[i])
        for i in range(len(OPENCLIP_LABELS))
    }

    best_label = max(scores, key=scores.get)
    return best_label, scores[best_label], scores


clip_records = []

for _, row in tqdm(df_layout.iterrows(), total=len(df_layout), desc="OpenCLIP classification"):
    best_label, best_score, scores = classify_openclip(row["image_path"])

    clip_records.append({
        **row.to_dict(),
        "clip_best_label": best_label,
        "clip_best_score": best_score,
        "clip_flowchart_score": scores[LABEL_FLOWCHART],
        "clip_workflow_score": scores[LABEL_WORKFLOW],
        "clip_decision_tree_score": scores[LABEL_DECISION_TREE],
        "clip_regulatory_flow_score": scores[LABEL_REGULATORY_FLOW],
        "clip_process_map_score": scores[LABEL_PROCESS_MAP],
        "clip_reference_map_score": scores[LABEL_REFERENCE_MAP],
        "clip_table_score": scores[LABEL_TABLE],
        "clip_text_page_score": scores[LABEL_TEXT_PAGE],
        "clip_logo_score": scores[LABEL_LOGO],
    })

df_clip = pd.DataFrame(clip_records)
print("OpenCLIP completed:", len(df_clip))

# ---------- geometry aliases / merged working frame ----------
df_clip_geo_layout = df_clip.copy()

# Alias names expected by later cells
df_clip_geo_layout["img_rectangle_like"] = df_clip_geo_layout.get("img_rect_like", 0)
df_clip_geo_layout["img_connector_like"] = df_clip_geo_layout.get("img_arrow_like", 0)

# Build a simple geometry_score from fast geometry signals if not already present
if "geometry_score" not in df_clip_geo_layout.columns:
    df_clip_geo_layout["geometry_score"] = (
        0.45 * df_clip_geo_layout.get("img_shape_score", 0) +
        0.20 * df_clip_geo_layout.get("img_line_score", 0) +
        0.15 * (df_clip_geo_layout.get("img_diamond_like", 0) > 0).astype(float) +
        0.10 * (df_clip_geo_layout.get("img_ellipse_like", 0) > 0).astype(float) +
        0.10 * (df_clip_geo_layout.get("img_arrow_like", 0) >= 2).astype(float)
    ).clip(0, 1)

preview_cols = [
    "page_number",
    "source_type",
    "clip_best_label",
    "clip_best_score",
    "clip_flowchart_score",
    "clip_workflow_score",
    "clip_decision_tree_score",
    "clip_regulatory_flow_score",
    "clip_process_map_score",
    "clip_reference_map_score",
    "clip_table_score",
    "clip_text_page_score",
    "clip_logo_score",
    "img_shape_score",
    "img_table_score",
    "layout_table_signal",
    "layout_figure_signal",
    "layout_text_signal",
    "layout_decision",
    "geometry_score",
]
preview_cols = [c for c in preview_cols if c in df_clip_geo_layout.columns]
display(df_clip_geo_layout[preview_cols].head(20))


Running Paddle Layout: 100%|██████████| 100/100 [00:00<00:00, 8790.51it/s]


Layout stage completed: 100


OpenCLIP classification: 100%|██████████| 100/100 [00:17<00:00,  5.81it/s]


OpenCLIP completed: 100


,page_number,source_type,clip_best_label,clip_best_score,clip_flowchart_score,clip_workflow_score,clip_decision_tree_score,clip_regulatory_flow_score,clip_process_map_score,clip_reference_map_score,clip_table_score,clip_text_page_score,clip_logo_score,img_shape_score,img_table_score,layout_table_signal,layout_figure_signal,layout_text_signal,layout_decision,geometry_score
0,357,rendered_vector_page,a document page with structured text,0.999909,4.266536e-08,6.105130e-06,7.234229e-07,4.427534e-05,2.721839e-07,7.883145e-07,0.000038,0.999909,2.374137e-08,1.0,0.3,0,0,0,neutral,1.0
1,339,embedded_image,a document page with structured text,0.918468,5.047565e-05,1.002000e-03,1.784620e-03,2.286281e-03,1.778575e-03,5.903888e-03,0.068391,0.918468,3.352415e-04,1.0,0.3,0,0,0,neutral,1.0
2,14,rendered_vector_page,a regulatory process flow diagram,0.715135,8.503988e-02,1.151090e-01,1.162594e-02,7.151348e-01,7.245738e-02,2.237330e-04,0.000003,0.000406,5.062733e-09,1.0,0.6,0,0,0,neutral,1.0
3,168,rendered_vector_page,a document page with structured text,0.620426,1.630563e-06,1.429054e-05,3.110866e-06,3.570487e-05,9.993180e-06,8.561099e-02,0.293895,0.620426,3.562227e-06,1.0,0.6,0,0,0,neutral,1.0
4,169,rendered_vector_page,a tabular document with rows and columns,0.501838,1.469273e-05,1.488152e-05,2.843146e-04,1.057447e-05,4.835527e-06,1.644698e-01,0.501838,0.333358,4.640818e-06,1.0,0.6,0,0,0,neutral,1.0
5,132,rendered_vector_page,a reference map or cross reference table,0.655931,3.173603e-05,1.170788e-05,2.061668e-04,2.949843e-04,1.776756e-06,6.559306e-01,0.149742,0.193780,1.366199e-06,1.0,0.6,0,0,0,neutral,1.0
6,167,rendered_vector_page,a reference map or cross reference table,0.418522,1.130043e-04,3.301941e-05,3.885269e-05,2.081775e-04,2.127219e-04,4.185222e-01,0.187913,0.392959,3.392164e-07,1.0,0.6,0,0,0,neutral,1.0
7,189,rendered_vector_page,a reference map or cross reference table,0.698104,2.573726e-05,4.998380e-05,1.249895e-05,1.045140e-04,2.299942e-05,6.981044e-01,0.194234,0.107445,1.901928e-07,1.0,0.6,0,0,0,neutral,1.0
8,194,rendered_vector_page,a reference map or cross reference table,0.527679,1.469772e-04,3.187956e-04,2.522429e-04,2.793647e-03,4.188116e-04,5.276791e-01,0.238044,0.230288,5.836019e-05,1.0,0.6,0,0,0,neutral,1.0
9,182,rendered_vector_page,a reference map or cross reference table,0.994741,2.714289e-07,9.433633e-06,3.705551e-06,4.034559e-06,1.612932e-06,9.947407e-01,0.004243,0.000998,9.149518e-09,1.0,0.6,0,0,0,neutral,1.0


In [12]:
# Cell 12: Score fusion + first-pass decision (main tuning cell)

def _safe(row, key, default=0.0):
    val = row.get(key, default)
    if val is None:
        return default
    return val


def score_fusion(row):
    """
    Combine:
    - rule / table penalties
    - Paddle layout
    - OpenCLIP semantics
    - Geometry signals

    Goal:
    high recall for true process/flow diagrams
    while down-ranking obvious tables/reference maps.
    """

    # ---------- semantic scores ----------
    clip_flowchart = _safe(row, "clip_flowchart_score", 0.0)
    clip_workflow = _safe(row, "clip_workflow_score", 0.0)
    clip_decision_tree = _safe(row, "clip_decision_tree_score", 0.0)
    clip_regulatory_flow = _safe(row, "clip_regulatory_flow_score", 0.0)

    clip_table = _safe(row, "clip_table_score", 0.0)
    clip_reference_map = _safe(row, "clip_reference_map_score", 0.0)

    semantic_positive = max(
        clip_flowchart,
        clip_workflow,
        clip_decision_tree,
        clip_regulatory_flow
    )
    semantic_negative = max(
        clip_table,
        clip_reference_map
    )

    # ---------- layout signals ----------
    layout_table_signal = _safe(row, "layout_table_signal", 0)
    layout_figure_signal = _safe(row, "layout_figure_signal", 0)
    layout_text_signal = _safe(row, "layout_text_signal", 0)
    layout_decision = str(row.get("layout_decision", "neutral"))

    # ---------- geometry ----------
    img_rect = _safe(row, "img_rect_like", 0)
    img_diamond = _safe(row, "img_diamond_like", 0)
    img_ellipse = _safe(row, "img_ellipse_like", 0)
    img_arrow = _safe(row, "img_arrow_like", 0)
    img_connector = _safe(row, "img_arrow_like", 0)
    geometry_score = _safe(row, "geometry_score", 0.0)

    # ---------- earlier rule signals ----------
    table_grid_score = _safe(row, "table_grid_score", 0.0)
    flow_word_count = _safe(row, "flow_word_count", 0)
    table_word_count = _safe(row, "table_word_count", 0)

    score = 0.0

    # =========================
    # POSITIVE SIGNALS
    # =========================
    score += 0.34 * semantic_positive
    score += 0.28 * geometry_score

    # Layout can boost if it sees diagram/image-like regions
    if layout_decision == "diagram_like_boost":
        score += 0.10
    if layout_figure_signal >= 1:
        score += 0.06

    # Geometry micro-boosts
    if img_rect >= 2:
        score += 0.06
    if img_diamond >= 1:
        score += 0.10
    if img_ellipse >= 1:
        score += 0.04
    if img_arrow >= 2:
        score += 0.10
    if img_connector >= 2:
        score += 0.06

    # keyword boosts are weak only
    if flow_word_count >= 2:
        score += 0.04
    if flow_word_count >= 5:
        score += 0.04

    # =========================
    # NEGATIVE SIGNALS
    # =========================
    score -= 0.18 * semantic_negative

    # table/layout penalties
    if layout_decision == "table_like_penalty":
        score -= 0.16

    if layout_table_signal >= 2:
        score -= 0.10
    if layout_text_signal >= 4 and layout_figure_signal == 0:
        score -= 0.08

    if table_grid_score >= 0.60:
        score -= 0.12
    if table_grid_score >= 0.80:
        score -= 0.10

    # keyword table penalty should be weak unless repeated
    if table_word_count >= 2:
        score -= 0.06
    if table_word_count >= 4:
        score -= 0.08

    # If semantic says table/reference-map but geometry is weak, penalize more
    if semantic_negative >= 0.45 and geometry_score < 0.25:
        score -= 0.10

    # clamp
    score = max(0.0, min(score, 1.0))
    return score


def first_pass_decision(row):
    """
    High-recall first-pass routing:
    - candidate
    - needs_review
    - reject

    We avoid aggressive reject unless multiple signals agree.
    """

    final_score = score_fusion(row)

    semantic_positive = max(
        _safe(row, "clip_flowchart_score", 0.0),
        _safe(row, "clip_workflow_score", 0.0),
        _safe(row, "clip_decision_tree_score", 0.0),
        _safe(row, "clip_regulatory_flow_score", 0.0)
    )
    semantic_negative = max(
        _safe(row, "clip_table_score", 0.0),
        _safe(row, "clip_reference_map_score", 0.0)
    )
    geometry_score = _safe(row, "geometry_score", 0.0)
    table_grid_score = _safe(row, "table_grid_score", 0.0)
    layout_decision = str(row.get("layout_decision", "neutral"))

    # strong reject only if several signals agree
    if (
        layout_decision == "table_like_penalty"
        and semantic_negative >= 0.55
        and geometry_score < 0.20
        and table_grid_score >= 0.60
    ):
        return "reject"

    # candidate
    if final_score >= 0.58:
        return "flowchart_candidate"

    # review bucket
    if final_score >= 0.34:
        return "needs_review"

    return "reject"


df_fused = df_clip_geo_layout.copy()

df_fused["fusion_score"] = df_fused.apply(score_fusion, axis=1)
df_fused["first_pass_decision"] = df_fused.apply(first_pass_decision, axis=1)

# keep alias names for later cells
df_fused["first_pass_score"] = df_fused["fusion_score"]

# review bucket for optional fallback
needs_paddle = df_fused[df_fused["first_pass_decision"] == "needs_review"].copy()

print(df_fused["first_pass_decision"].value_counts())

preview_cols = [
    "page_number",
    "source_type",
    "fusion_score",
    "first_pass_score",
    "first_pass_decision",
    "layout_decision",
    "layout_table_signal",
    "layout_figure_signal",
    "clip_flowchart_score",
    "clip_workflow_score",
    "clip_regulatory_flow_score",
    "clip_table_score",
    "clip_reference_map_score",
    "geometry_score"
]
preview_cols = [c for c in preview_cols if c in df_fused.columns]
display(df_fused[preview_cols].head(30))

first_pass_decision
reject                 65
needs_review           27
flowchart_candidate     8
Name: count, dtype: int64


,page_number,source_type,fusion_score,first_pass_score,first_pass_decision,layout_decision,layout_table_signal,layout_figure_signal,clip_flowchart_score,clip_workflow_score,clip_regulatory_flow_score,clip_table_score,clip_reference_map_score,geometry_score
0,357,rendered_vector_page,0.520008,0.520008,needs_review,neutral,0,0,4.266536e-08,6.105130e-06,4.427534e-05,0.000038,7.883145e-07,1.000
1,339,embedded_image,0.608467,0.608467,flowchart_candidate,neutral,0,0,5.047565e-05,1.002000e-03,2.286281e-03,0.068391,5.903888e-03,1.000
2,14,rendered_vector_page,0.863106,0.863106,flowchart_candidate,neutral,0,0,8.503988e-02,1.151090e-01,7.151348e-01,0.000003,2.237330e-04,1.000
3,168,rendered_vector_page,0.387111,0.387111,needs_review,neutral,0,0,1.630563e-06,1.429054e-05,3.570487e-05,0.293895,8.561099e-02,1.000
4,169,rendered_vector_page,0.349766,0.349766,needs_review,neutral,0,0,1.469273e-05,1.488152e-05,1.057447e-05,0.501838,1.644698e-01,1.000
5,132,rendered_vector_page,0.362033,0.362033,needs_review,neutral,0,0,3.173603e-05,1.170788e-05,2.949843e-04,0.149742,6.559306e-01,1.000
6,167,rendered_vector_page,0.404737,0.404737,needs_review,neutral,0,0,1.130043e-04,3.301941e-05,2.081775e-04,0.187913,4.185222e-01,1.000
7,189,rendered_vector_page,0.354377,0.354377,needs_review,neutral,0,0,2.573726e-05,4.998380e-05,1.045140e-04,0.194234,6.981044e-01,1.000
8,194,rendered_vector_page,0.345968,0.345968,needs_review,neutral,0,0,1.469772e-04,3.187956e-04,2.793647e-03,0.238044,5.276791e-01,1.000
9,182,rendered_vector_page,0.260950,0.260950,reject,neutral,0,0,2.714289e-07,9.433633e-06,4.034559e-06,0.004243,9.947407e-01,1.000


In [13]:
!pip uninstall -y transformers tokenizers -q
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "sentencepiece" "einops" "timm" "accelerate"

In [16]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch

FLORENCE_MODEL_ID = "microsoft/Florence-2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Step 1: importing ok")

processor = AutoProcessor.from_pretrained(
    FLORENCE_MODEL_ID,
    trust_remote_code=True
)
print("Step 2: processor loaded")

model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=dtype
).to(device)

model.eval()
print("Step 3: model loaded on", device)

Step 1: importing ok
Step 2: processor loaded


ImportError: This modeling file requires the following packages that were not found in your environment: flash_attn. Run `pip install flash_attn`

In [15]:
# Florence-2 pipeline test cell

import torch
from PIL import Image
from transformers import pipeline

FLORENCE_MODEL_ID = "microsoft/Florence-2-base"
device = 0 if torch.cuda.is_available() else -1

florence_pipe = pipeline(
    task="image-text-to-text",
    model=FLORENCE_MODEL_ID,
    trust_remote_code=True,
    device=device
)

print("Florence pipeline loaded")

KeyError: "Unknown task image-text-to-text, available tasks are ['audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-to-image', 'image-to-text', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'summarization', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'text2text-generation', 'token-classification', 'translation', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
# Florence-2 single-image test

test_image_path = needs_paddle.iloc[0]["image_path"]  # or give any image path manually

prompt = (
    "Classify this image into exactly one label: "
    "flowchart_like, table_like, reference_map_like, other. "
    "Return only the label and one short reason."
)

image = Image.open(test_image_path).convert("RGB")

result = florence_pipe(image, text=prompt, max_new_tokens=32)
print(result)

In [ ]:
# Cell 13: Florence-2 fallback only for uncertain pages
# Pipeline-based version

fallback_results = []

if USE_FLORENCE_FALLBACK and len(needs_paddle) > 0:
    try:
        import torch
        from PIL import Image
        from transformers import pipeline

        florence_device = 0 if torch.cuda.is_available() else -1

        florence_pipe = pipeline(
            task="image-text-to-text",
            model=FLORENCE_MODEL_ID,
            trust_remote_code=True,
            device=florence_device
        )

        fallback_input = (
            needs_paddle
            .sort_values("fusion_score", ascending=False)
            .head(MAX_FLORENCE_ITEMS)
            .copy()
        )

        classification_prompt = (
            "Classify this image into exactly one label: "
            "flowchart_like, table_like, reference_map_like, other. "
            "Return only the label and one short reason."
        )

        def run_florence_prompt(image_path, prompt):
            image = Image.open(image_path).convert("RGB")
            output = florence_pipe(image, text=prompt, max_new_tokens=32)
            if isinstance(output, list) and len(output) > 0:
                item = output[0]
                if isinstance(item, dict):
                    return str(
                        item.get("generated_text")
                        or item.get("text")
                        or item
                    )
            return str(output)

        for _, row in tqdm(
            fallback_input.iterrows(),
            total=len(fallback_input),
            desc="Florence-2 fallback"
        ):
            try:
                generated_text = run_florence_prompt(row["image_path"], classification_prompt)
                text_blob = str(generated_text).lower()
            except Exception as e:
                text_blob = f"florence_error: {e}"

            if "flowchart_like" in text_blob:
                predicted_label = "flowchart_like"
            elif "reference_map_like" in text_blob:
                predicted_label = "reference_map_like"
            elif "table_like" in text_blob:
                predicted_label = "table_like"
            else:
                predicted_label = "other"

            flow_signal = count_keywords(text_blob, FLOW_WORDS)
            table_signal = count_keywords(text_blob, TABLE_WORDS)
            negative_signal = count_keywords(text_blob, NEGATIVE_VISUAL_WORDS)

            if predicted_label == "flowchart_like":
                flow_signal += 2
            elif predicted_label in ["table_like", "reference_map_like"]:
                table_signal += 2

            fallback_results.append({
                "page_number": row["page_number"],
                "image_path": row["image_path"],
                "fallback_label": predicted_label,
                "fallback_flow_signal": flow_signal,
                "fallback_table_signal": table_signal,
                "fallback_negative_signal": negative_signal,
                "fallback_text_preview": text_blob[:500],
            })

    except Exception as e:
        print("Florence-2 unavailable or failed to load:", e)
else:
    print("Skipping Florence-2 fallback.")

df_fallback = pd.DataFrame(fallback_results)
print("Florence fallback completed:", len(df_fallback))
display(df_fallback.head(20))

In [ ]:
# Cell 14: Final decision after optional fallback

def final_decision_after_fallback(row):
    """
    Final stage:
    - keep high-recall behavior
    - let fallback rescue difficult true flow diagrams
    - only hard reject when multiple negative signals agree
    """

    first_pass = str(row.get("first_pass_decision", "reject"))
    fusion_score = _safe(row, "fusion_score", 0.0)

    fallback_flow = _safe(row, "fallback_flow_signal", 0)
    fallback_table = _safe(row, "fallback_table_signal", 0)
    fallback_label = str(row.get("fallback_label", ""))

    semantic_negative = max(
        _safe(row, "clip_table_score", 0.0),
        _safe(row, "clip_reference_map_score", 0.0)
    )
    geometry_score = _safe(row, "geometry_score", 0.0)
    layout_decision = str(row.get("layout_decision", "neutral"))
    table_grid_score = _safe(row, "table_grid_score", 0.0)

    # already good
    if first_pass == "flowchart_candidate":
        return "flowchart_candidate"

    # rescue by fallback
    if first_pass == "needs_review":
        if fallback_label == "flowchart_like" and fallback_table == 0:
            return "flowchart_candidate"

        if fallback_flow >= 2 and fallback_table == 0:
            return "flowchart_candidate"

        if fallback_flow >= 1 and fusion_score >= 0.42:
            return "flowchart_candidate"

        if fallback_label in ["table_like", "reference_map_like"] and fallback_flow == 0:
            return "reject"

        if fallback_table >= 2 and fallback_flow == 0:
            return "reject"

        return "needs_review"

    # reject bucket can still be rescued if fallback strongly supports flow
    if first_pass == "reject":
        if fallback_label == "flowchart_like" and geometry_score >= 0.20:
            return "needs_review"

        if fallback_flow >= 3 and fallback_table == 0 and geometry_score >= 0.20:
            return "needs_review"

        # hard reject only when all negatives align
        if (
            layout_decision == "table_like_penalty"
            and semantic_negative >= 0.60
            and geometry_score < 0.20
            and table_grid_score >= 0.60
        ):
            return "reject"

        return "reject"

    return "reject"


df_final = df_fused.copy()

if "df_fallback" in globals() and len(df_fallback) > 0:
    df_final = df_final.merge(
        df_fallback[[
            "page_number",
            "image_path",
            "fallback_label",
            "fallback_flow_signal",
            "fallback_table_signal"
        ]],
        on=["page_number", "image_path"],
        how="left"
    )
else:
    df_final["fallback_label"] = ""
    df_final["fallback_flow_signal"] = 0
    df_final["fallback_table_signal"] = 0

df_final["fallback_flow_signal"] = df_final["fallback_flow_signal"].fillna(0)
df_final["fallback_table_signal"] = df_final["fallback_table_signal"].fillna(0)
df_final["fallback_label"] = df_final["fallback_label"].fillna("")

df_final["final_decision"] = df_final.apply(final_decision_after_fallback, axis=1)

print(df_final["final_decision"].value_counts())

display(df_final[[
    "page_number",
    "source_type",
    "fusion_score",
    "first_pass_decision",
    "fallback_label",
    "fallback_flow_signal",
    "fallback_table_signal",
    "final_decision"
]].head(40))

# compatibility aliases for display cell
df_final["final_score"] = df_final["fusion_score"]
df_final["first_pass_score"] = df_final["first_pass_score"] if "first_pass_score" in df_final.columns else df_final["fusion_score"]


In [ ]:
# Cell 15: Display flowchart candidates
# Displays only final flowchart candidates. Increase DISPLAY_LIMIT if needed.

DISPLAY_LIMIT = 40

if df_final.empty:
    print("No final results available.")
else:
    flowcharts = df_final[df_final["final_decision"] == "flowchart_candidate"].sort_values("final_score", ascending=False).copy()
    print("Flowchart candidates:", len(flowcharts))

    preview_cols = [
        "page_number", "source_type", "final_score", "first_pass_score",
        "clip_best_label", "clip_flowchart_score", "clip_workflow_score", "clip_table_score",
        "img_shape_score", "img_table_score", "table_grid_score", "flow_word_count", "table_word_count"
    ]
    preview_cols = [c for c in preview_cols if c in flowcharts.columns]
    display(flowcharts[preview_cols].head(DISPLAY_LIMIT))

    for _, row in flowcharts.head(DISPLAY_LIMIT).iterrows():
        print("=" * 80)
        print(f"Page: {row['page_number']} | Source: {row['source_type']} | Final score: {row['final_score']:.3f}")
        print(f"Image: {row['image_path']}")
        display(Image.open(row["image_path"]))
